In [22]:
import pandas as pd
import numpy as np
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.metrics import (
    classification_report
)

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from xgboost import XGBClassifier

# SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [23]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1329, 9)
Test Shape: (333, 9)


In [24]:
categorical_cols = [
    "ip_type",
    "source"
]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

Numerical Columns:
['inter_api_access_duration(sec)', 'api_access_uniqueness', 'sequence_length(count)', 'vsession_duration(min)', 'num_sessions', 'num_users', 'num_unique_apis']

Categorical Columns:
['ip_type', 'source']


In [25]:
preprocessor = ColumnTransformer([

    (
        "num",

        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),

        numerical_cols
    ),

    (
        "cat",

        Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),

            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore",  sparse_output=False)
            )
        ]),

        categorical_cols
    )
])

In [26]:
models = {

    "RandomForest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            use_label_encoder=False
        ),

    "SVM":
        SVC(),

    "LogisticRegression":
        LogisticRegression(
            max_iter=1000
        ),

    "NaiveBayes":
        GaussianNB()
}

In [27]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro"
}

In [28]:
results = []

for model_name, model in models.items():

    print(f"\n🚀 Training: {model_name}")

    # FULL PIPELINE
    pipeline = ImbPipeline([

        ("preprocessor", preprocessor),

        # SMOTE AFTER SCALING
        ("smote", SMOTE(random_state=42)),

        ("model", model)
    ])

    # Cross Validation
    scores = cross_validate(

        pipeline,

        X_train,
        y_train,

        cv=skf,

        scoring=scoring,

        n_jobs=-1,

        return_train_score=False
    )

    # Store results
    results.append({

        "Model": model_name,

        "Accuracy":
            round(scores["test_accuracy"].mean(), 4),

        "Precision":
            round(scores["test_precision"].mean(), 4),

        "Recall":
            round(scores["test_recall"].mean(), 4),

        "F1-Score":
            round(scores["test_f1"].mean(), 4)
    })

    print("✅ Done")


🚀 Training: RandomForest
✅ Done

🚀 Training: XGBoost
✅ Done

🚀 Training: SVM
✅ Done

🚀 Training: LogisticRegression
✅ Done

🚀 Training: NaiveBayes
✅ Done


In [29]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1-Score",
    ascending=False
)

print("\n📊 FINAL RESULTS\n")

print(results_df)


📊 FINAL RESULTS

                Model  Accuracy  Precision  Recall  F1-Score
0        RandomForest    1.0000     1.0000  1.0000    1.0000
2                 SVM    1.0000     1.0000  1.0000    1.0000
1             XGBoost    0.9992     0.9989  0.9994    0.9992
3  LogisticRegression    0.9992     0.9994  0.9989    0.9992
4          NaiveBayes    0.9985     0.9989  0.9978    0.9983


In [30]:
os.makedirs(os.path.join('..', 'reports'), exist_ok=True)
results_df.to_csv(
    "../reports/model_results.csv",
    index=False
)

print("✅ Results Saved")

✅ Results Saved


In [31]:
best_pipeline = ImbPipeline([

    ("preprocessor", preprocessor),

    ("smote", SMOTE(random_state=42)),

    (
        "model",

        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        )
    )
])

best_pipeline.fit(X_train, y_train)

print("✅ Final Model Trained")

✅ Final Model Trained


In [32]:
y_pred = best_pipeline.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       222
           1       1.00      1.00      1.00       111

    accuracy                           1.00       333
   macro avg       1.00      1.00      1.00       333
weighted avg       1.00      1.00      1.00       333



In [33]:
import joblib
import os

joblib.dump(
    best_pipeline,
    "../model/final_model.pkl"
)

print("✅ Model Saved")

✅ Model Saved
